# 03. Custom Validators & Computed Fields

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/pydantic/en/03_validators_and_computed.ipynb)

Built-in `Field(...)` constraints cover numeric bounds, lengths, and regex patterns, while custom Python domain logic hooks into four distinct stages: pre-parsing normalization (`@field_validator(..., mode="before")`), typed single-field checks (`@field_validator(..., mode="after")`), cross-field invariants (`@model_validator(mode="after")`), and derived serialization properties (`@computed_field`).
Understanding the exact execution order of these hooks lets Pydantic v2's Rust core handle fast type coercion while layering domain invariants cleanly before, after, and during JSON export.

![Figure 3-1: Pydantic v2 Validation & Computed Field Execution Order](https://examples.drawlib.com/pydantic/images/03_validators_and_computed/ch03_validator_lifecycle.png)

*Figure 3-1: Pydantic v2 Validation & Computed Field Execution Order*

---

## 1. Normalizing & Checking Fields with `@field_validator`

`@field_validator` attaches custom validation or normalization logic to one or more named fields on a `BaseModel`. Always place `@classmethod` immediately beneath `@field_validator(...)` and choose the execution `mode` based on whether your code expects raw input or an already-parsed Python type:

- **`mode="before"` (Raw Input Normalization)**: Runs *before* Pydantic's Rust core parses the field. Because incoming data could be any raw JSON or form value (`object`), guard type assumptions with `isinstance(...)`—for example, splitting a comma-separated string `" python, fastapi , ai "` into `["python", "fastapi", "ai"]` while letting native `list[str]` inputs pass straight through.
- **`mode="after"` (Default — Typed Single-Field Validation)**: Runs *after* Rust core validation succeeds, guaranteeing that `value` already matches the annotated Python type (`str`). Raise a `ValueError` with a clear message whenever a domain rule fails; Pydantic automatically wraps it into a structured `ValidationError`.

In [ ]:
!pip install -q pydantic

In [ ]:
from pydantic import BaseModel, ValidationError, field_validator

RESERVED_USERNAMES = {"admin", "root", "system"}


class UserProfile(BaseModel):
    username: str
    tags: list[str] = []

    @field_validator("tags", mode="before")
    @classmethod
    def split_csv_tags(cls, value: object) -> object:
        if isinstance(value, str):
            return [tag.strip().lower() for tag in value.split(",") if tag.strip()]
        return value

    @field_validator("username", mode="after")
    @classmethod
    def normalize_and_check_username(cls, value: str) -> str:
        cleaned = value.strip().lower()
        if not cleaned:
            raise ValueError("username cannot be blank")
        if cleaned in RESERVED_USERNAMES:
            raise ValueError(f"'{cleaned}' is a reserved username")
        return cleaned


profile = UserProfile(username="  Alice_Dev  ", tags=" python, fastapi , ai ")
print("Normalized profile:", profile)
print("Dumped dict:", profile.model_dump())

try:
    UserProfile(username="Admin", tags=["ops", "security"])
except ValidationError as err:
    print("\nRejected reserved username:")
    for issue in err.errors():
        print(f"  loc={issue['loc']} | type={issue['type']} | msg={issue['msg']}")

> **Note:** Raise `ValueError` (or `AssertionError`) inside validators rather than instantiating `ValidationError` directly. Pydantic v2 catches `ValueError` and enriches it with the exact field path (`loc`), input value, and `value_error` metadata.

---

## 2. Cross-Field Rules (`@model_validator`) & Derived `@computed_field`s

Single-field validators operate in isolation—they should not depend on sibling fields that might have failed validation or been declared later. When a rule spans multiple fields, or when downstream APIs need derived attributes serialized into JSON, reach for `@model_validator(mode="after")` and `@computed_field`:

- **`@model_validator(mode="after")`**: Runs as an instance method receiving `self` once **all** individual fields have passed validation and type coercion (for example, `check_in` and `check_out` are already `datetime.date` objects). Enforce cross-field invariants like `self.check_out > self.check_in` and always `return self`.
- **`@computed_field` + `@property`**: Whereas a plain Python `@property` is accessible in Python code (`booking.nights`) yet silently omitted during serialization, stacking `@computed_field` above `@property` includes the computed value automatically in both `model_dump()` and `model_dump_json()`.

![Figure 3-2: Single-Field vs. Cross-Field Validation and @computed_field Serialization](https://examples.drawlib.com/pydantic/images/03_validators_and_computed/ch03_cross_field_and_computed.png)

*Figure 3-2: Single-Field vs. Cross-Field Validation and @computed_field Serialization*

### Validating Date Ranges & Serializing Computed Totals

Let's combine `@model_validator(mode="after")` and `@computed_field` in a `HotelBooking` model. Notice how ISO date strings (`"2026-05-10"`) are parsed into `datetime.date` instances first, checked for `check_out > check_in`, and serialized with `nights` and `total_price` included automatically in the JSON payload:

In [ ]:
from datetime import date
from typing import Self

from pydantic import BaseModel, Field, ValidationError, computed_field, model_validator


class HotelBooking(BaseModel):
    check_in: date
    check_out: date
    nightly_rate: int = Field(gt=0)

    @model_validator(mode="after")
    def check_date_order(self) -> Self:
        if self.check_out <= self.check_in:
            raise ValueError("check_out must be strictly after check_in")
        return self

    @computed_field
    @property
    def nights(self) -> int:
        return (self.check_out - self.check_in).days

    @computed_field
    @property
    def total_price(self) -> int:
        return self.nights * self.nightly_rate


booking = HotelBooking(
    check_in="2026-05-10",
    check_out="2026-05-14",
    nightly_rate=180,
)
print("Serialized JSON with @computed_field values:")
print(booking.model_dump_json(indent=2))

try:
    HotelBooking(
        check_in="2026-05-14",
        check_out="2026-05-10",
        nightly_rate=180,
    )
except ValidationError as err:
    print("\nCaught invalid date range:")
    for issue in err.errors():
        print(f"  loc={issue['loc']} | type={issue['type']} | msg={issue['msg']}")

### Key Takeaways

- **Normalize Raw Payloads in `mode="before"`, Validate Types in `mode="after"`**: Let `pydantic-core` perform type coercion in between so `mode="after"` validators receive guaranteed Python types.
- **Enforce Multi-Field Invariants in `@model_validator(mode="after")`**: Validate relationships like `check_out > check_in` against the fully populated `self` instance and always `return self`.
- **Expose Derived Fields via `@computed_field`**: Stack `@computed_field` above `@property` whenever calculated values such as `nights` and `total_price` belong in `model_dump()` and JSON API responses.